# 04 - Data Preprocessing & Feature Engineering



In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as ssum, when


In [2]:
spark = ( SparkSession.builder .appName("Bus_Risk_Preprocessing") 
    .master("local[*]") 
    .config("spark.driver.host", "127.0.0.1") 
    .config("spark.driver.bindAddress", "127.0.0.1") 
    .config("spark.local.ip", "127.0.0.1") 
    .config("spark.driver.memory", "4g") 
    .config("spark.python.worker.reuse", "true") 
    .config("spark.sql.shuffle.partitions", "16") 
    .config("spark.default.parallelism", "16") 
    .getOrCreate() )

spark.sparkContext.setLogLevel("ERROR")
print("Cores available to Spark:", spark.sparkContext.defaultParallelism)


Cores available to Spark: 16


In [3]:
project_path = r"D:\BigDataCoursework"
processed_path = os.path.join(project_path, "Data", "processed")

journeys_df = spark.read.parquet(os.path.join(processed_path, "integrated_journeys.parquet"))
journeys_df.printSchema()


root
 |-- Journey_ID: string (nullable = true)
 |-- Operator_ID: string (nullable = true)
 |-- Service_Code: string (nullable = true)
 |-- Line_Name: string (nullable = true)
 |-- Departure_Hour: integer (nullable = true)
 |-- Peak_Hour: integer (nullable = true)
 |-- Route_Complexity: long (nullable = true)
 |-- Number_of_Stops: long (nullable = true)
 |-- Risk_Score: double (nullable = true)
 |-- Service_Risk: integer (nullable = true)
 |-- Disruption_Count: long (nullable = true)
 |-- Has_Disruption: integer (nullable = true)
 |-- Position_Reports: long (nullable = true)
 |-- Has_Live_Tracking: integer (nullable = true)



## Partitioning, caching & parallelism evidence

In [4]:
print("Initial Partitions:", journeys_df.rdd.getNumPartitions())

journeys_df = journeys_df.repartition(max(4, spark.sparkContext.defaultParallelism))
journeys_df.cache()
journeys_df.count()  # materialise cache

print("Partitions after repartition:", journeys_df.rdd.getNumPartitions())
print("Cores used (defaultParallelism):", spark.sparkContext.defaultParallelism)


Initial Partitions: 16
Partitions after repartition: 16
Cores used (defaultParallelism): 16


In [5]:
print("Total rows:", journeys_df.count())
print("Unique Journey IDs:", journeys_df.select("Journey_ID").distinct().count())
print("Unique Journey + Operator combos:", journeys_df.select("Journey_ID", "Operator_ID").distinct().count())


Total rows: 227010
Unique Journey IDs: 1934
Unique Journey + Operator combos: 3086


In [6]:
journeys_df.select([
    ssum(col(c).isNull().cast("int")).alias(c) for c in journeys_df.columns
]).show()


+----------+-----------+------------+---------+--------------+---------+----------------+---------------+----------+------------+----------------+--------------+----------------+-----------------+
|Journey_ID|Operator_ID|Service_Code|Line_Name|Departure_Hour|Peak_Hour|Route_Complexity|Number_of_Stops|Risk_Score|Service_Risk|Disruption_Count|Has_Disruption|Position_Reports|Has_Live_Tracking|
+----------+-----------+------------+---------+--------------+---------+----------------+---------------+----------+------------+----------------+--------------+----------------+-----------------+
|         0|          0|           0|        0|             0|        0|               0|              0|         0|           0|               0|             0|               0|                0|
+----------+-----------+------------+---------+--------------+---------+----------------+---------------+----------+------------+----------------+--------------+----------------+-----------------+



## Feature assembly 

In [7]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "Route_Complexity", "Number_of_Stops", "Departure_Hour", "Peak_Hour",
    "Disruption_Count", "Has_Disruption", "Position_Reports", "Has_Live_Tracking",
]

ml_df = journeys_df.select(*feature_cols, "Service_Risk").na.drop()

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
ml_data = assembler.transform(ml_df)

ml_data.select("features", "Service_Risk").show(5, truncate=False)


+---------------------------------------+------------+
|features                               |Service_Risk|
+---------------------------------------+------------+
|[658.0,208.0,16.0,1.0,0.0,0.0,22.0,1.0]|2           |
|[7512.0,28.0,17.0,1.0,48.0,1.0,0.0,0.0]|1           |
|[658.0,229.0,9.0,1.0,0.0,0.0,22.0,1.0] |2           |
|[451.0,154.0,8.0,1.0,0.0,0.0,22.0,1.0] |2           |
|[1060.0,15.0,16.0,1.0,0.0,0.0,41.0,1.0]|0           |
+---------------------------------------+------------+
only showing top 5 rows



In [8]:
feature_path = os.path.join(processed_path, "ml_features")
ml_data.write.mode("overwrite").parquet(feature_path)

print("Feature engineering completed!")
print("Saved to:", feature_path)
print("Total Records:", ml_data.count())


Feature engineering completed!
Saved to: D:\BigDataCoursework\Data\processed\ml_features
Total Records: 227010


In [9]:
train_data, test_data = ml_data.randomSplit([0.8, 0.2], seed=42)

print("Training Records:", train_data.count())
print("Testing Records:", test_data.count())


Training Records: 181952
Testing Records: 45058


In [10]:
ml_path = os.path.join(processed_path, "ml_data")

train_data.write.mode("overwrite").parquet(os.path.join(ml_path, "train"))
test_data.write.mode("overwrite").parquet(os.path.join(ml_path, "test"))

print("ML train/test datasets saved successfully")


ML train/test datasets saved successfully


In [11]:
journeys_df.groupBy("Service_Risk").count().show()
journeys_df.groupBy("Has_Disruption").count().show()


+------------+------+
|Service_Risk| count|
+------------+------+
|           1|101722|
|           2| 71046|
|           0| 54242|
+------------+------+

+--------------+------+
|Has_Disruption| count|
+--------------+------+
|             1| 57125|
|             0|169885|
+--------------+------+



In [12]:
journeys_output = os.path.join(processed_path, "journeys.parquet")
journeys_df.write.mode("overwrite").parquet(journeys_output)
print("Journeys dataset saved successfully")


Journeys dataset saved successfully


In [13]:
journeys_df.unpersist()
spark.stop()
print("Spark stopped successfully.")


Spark stopped successfully.
